In [15]:
import boto3
import sagemaker
from sagemaker.estimator import Estimator

# Sesión SageMaker
sess = sagemaker.Session()
role = sagemaker.get_execution_role()

# Región y cuenta AWS
region = boto3.Session().region_name or "us-east-1"
account_id = boto3.client("sts").get_caller_identity()["Account"]

# Bucket por defecto de SageMaker
default_bucket = sess.default_bucket()

# Nombre de la imagen en ECR
image_name = "ridge-sales"
image_uri = f"{account_id}.dkr.ecr.{region}.amazonaws.com/{image_name}:latest"

# Rutas S3
train_s3_uri = f"s3://{default_bucket}/granescala/train/"
output_s3_uri = f"s3://{default_bucket}/granescala/output"

print("role:", role)
print("region:", region)
print("account_id:", account_id)
print("default_bucket:", default_bucket)
print("image_uri:", image_uri)
print("train_s3_uri:", train_s3_uri)
print("output_s3_uri:", output_s3_uri)

estimator = Estimator(
    image_uri=image_uri,
    role=role,
    instance_count=1,
    instance_type="ml.m5.large",
    output_path=output_s3_uri,
    sagemaker_session=sess,
    hyperparameters={
        "alpha": 1.0,
        "monthly-file": "monthly.pkl",
        "base-file": "base.pkl",
    },
)

estimator.fit({
    "train": train_s3_uri
})

INFO:sagemaker.telemetry.telemetry_logging:SageMaker Python SDK will collect telemetry to help us better understand our user's needs, diagnose issues, and deliver additional features.
To opt out of telemetry, please disable via TelemetryOptOut parameter in SDK defaults config. For more information, refer to https://sagemaker.readthedocs.io/en/stable/overview.html#configuring-and-using-defaults-with-the-sagemaker-python-sdk.
INFO:sagemaker:Creating training-job with name: ridge-sales-2026-03-10-04-46-27-274


role: arn:aws:iam::613782679215:role/SageMakerStudioExecutionRole2026
region: us-east-1
account_id: 613782679215
default_bucket: sagemaker-us-east-1-613782679215
image_uri: 613782679215.dkr.ecr.us-east-1.amazonaws.com/ridge-sales:latest
train_s3_uri: s3://sagemaker-us-east-1-613782679215/granescala/train/
output_s3_uri: s3://sagemaker-us-east-1-613782679215/granescala/output
2026-03-10 04:46:27 Starting - Starting the training job...
2026-03-10 04:46:53 Starting - Preparing the instances for training...
2026-03-10 04:47:16 Downloading - Downloading input data...
2026-03-10 04:48:01 Training - Training image download completed. Training in progress...<frozen runpy>:128: RuntimeWarning: 'src.training.train' found in sys.modules after import of package 'src.training', but prior to execution of 'src.training.train'; this may result in unpredictable behaviour
2026-03-10 04:48:07,104 - train - INFO - Logger inicializado. Archivo: /opt/program/artifacts/logs/train_20260310_044807.log
2026-03-

In [ ]:
predictor = estimator.deploy(
    initial_instance_count=1,
    instance_type="ml.m5.large",
    endpoint_name="ridge-sales-endpoint"
)

INFO:sagemaker:Creating model with name: ridge-sales-2026-03-10-04-49-01-222
INFO:sagemaker:Creating endpoint-config with name ridge-sales-endpoint
INFO:sagemaker:Creating endpoint with name ridge-sales-endpoint


--------------------------------

In [ ]:
from sagemaker.serializers import JSONSerializer
from sagemaker.deserializers import JSONDeserializer

predictor.serializer = JSONSerializer()
predictor.deserializer = JSONDeserializer()

payload = {
    "instances": [
        {
            "shop_id": 31,
            "item_id": 456,
            "item_category_id": 12,
            "month": 11,
            "lag1_cnt": 2.0,
            "lag12_cnt": 1.0,
            "avg_price": 399.0
        },
        {
            "shop_id": 15,
            "item_id": 789,
            "item_category_id": 6,
            "month": 11,
            "lag1_cnt": 0.0,
            "lag12_cnt": 3.0,
            "avg_price": 199.0
        }
    ]
}

response = predictor.predict(payload)
print(response)